In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from dataclasses import dataclass
from typing import Optional, Dict, Any



In [2]:
# Preguntas

# ¿Time Cold: cuanto tiempo dejo una orden si no se ejecuta?
# ¿Que pasa si tengo 2 ordenes puestas? 

# Stop loss y Take profit 
# 1:2 --> SL = -1.5% de caida y TP: 3%
# ATR

## Filtro de eventos CUSUM para el ambiente de trading

El filtro **CUSUM** (*Cumulative Sum*) se usa como una etapa de preprocesamiento para detectar momentos en los que el precio acumuló un movimiento suficientemente grande frente a un umbral.

En este notebook conviene usarlo como **feature adicional** del ambiente, no necesariamente como filtro para borrar filas. Si eliminamos las velas que no son evento, el ambiente deja de avanzar en una línea temporal continua y los `step()` representan saltos entre eventos. Para Reinforcement Learning suele ser más estable conservar toda la serie y agregar una columna binaria `cusum_event` que indique cuándo ocurrió un evento.

Interpretación:

- `cusum_event = 1`: el precio acumuló un movimiento relevante y puede ser un momento interesante para abrir, cerrar o ajustar una posición.
- `cusum_event = 0`: no hubo un cambio acumulado suficiente según el umbral definido.

En datos reales, el umbral recomendado puede ser dinámico, por ejemplo `0.5 * ATR`, para que el filtro se adapte a la volatilidad del mercado.

In [ ]:
def cusum_filter(close, threshold):
    """
    Detecta eventos de mercado usando un filtro CUSUM simétrico.

    Parámetros
    ----------
    close : array-like
        Serie de precios de cierre. En este ambiente puede ser el vector `prices`.

    threshold : float o array-like
        Umbral mínimo de movimiento acumulado requerido para disparar un evento.
        - Si es float, se usa el mismo umbral en toda la serie.
        - Si es array-like, permite usar un umbral dinámico, por ejemplo 0.5 * ATR.

    Retorna
    -------
    np.ndarray
        Índices donde ocurrió un evento CUSUM.

    Cómo interpretarlo
    ------------------
    El filtro acumula cambios positivos y negativos del precio:
    - Si la suma positiva supera `threshold`, registra un evento alcista.
    - Si la suma negativa cae por debajo de `-threshold`, registra un evento bajista.

    El evento no es una señal directa de compra o venta. Solo indica que hubo
    suficiente movimiento acumulado para que el agente lo considere dentro de
    su observación o lógica de decisión.
    """
    close = np.asarray(close, dtype=np.float64)

    if close.ndim != 1:
        raise ValueError("close debe ser un vector unidimensional de precios")

    if np.isscalar(threshold):
        threshold_values = np.full(close.shape, float(threshold), dtype=np.float64)
    else:
        threshold_values = np.asarray(threshold, dtype=np.float64)
        if threshold_values.shape[0] != close.shape[0]:
            raise ValueError("threshold debe ser escalar o tener la misma longitud que close")

    t_events = []
    s_pos, s_neg = 0.0, 0.0
    price_diff = np.diff(close, prepend=np.nan)

    for i in range(1, len(close)):
        th = threshold_values[i]

        # Saltamos observaciones no válidas o umbrales no positivos.
        if np.isnan(th) or th <= 0 or np.isnan(price_diff[i]):
            continue

        # Acumulador de movimientos positivos y negativos.
        s_pos = max(0.0, s_pos + price_diff[i])
        s_neg = min(0.0, s_neg + price_diff[i])

        # Si se supera el umbral positivo o negativo, registramos evento
        # y reseteamos el acumulador correspondiente.
        if s_pos > th:
            s_pos = 0.0
            t_events.append(i)
        elif s_neg < -th:
            s_neg = 0.0
            t_events.append(i)

    return np.asarray(t_events, dtype=np.int64)

In [ ]:
# tomar en cuenta budget y quantity en state

In [3]:
atr = 0.03
entry_price = 100
print((1-atr14)*entry_price,  (1+0.95*atr14)*entry_price)

98.5 103.0


In [ ]:
@dataclass
class Position:
    side: int = 0              # 0 = flat, 1 = long, -1 = short
    quantity: float = 0.0      # kelly
    entry_price: float = 0.0   # depender de precio actual y slippage
    stop_loss_price: float = 0.0 #  (1-atr)*entry_price 
    take_profit_price: float = 0.0 # (1+atr)*entry_price
    max_duration: int = 0 # ??

In [2]:
@dataclass
class Position:
    side: int = 0              # 0 = flat, 1 = long, -1 = short

In [3]:
class TradingRLEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    HOLD = 0
    BUY = 1
    SELL = 2

    def __init__(
        self,
        data: np.ndarray,
        prices: np.ndarray,
        initial_budget: float = 10000.0,
        trade_fee: float = 0.001,
        max_episode_steps: Optional[int] = None,
        allow_short: bool = True,
        reward_scaling: float = 1.0,
        drawdown_penalty: float = 0.0,
        unrealized_pnl_weight: float = 0.0,
        min_stop_loss: float = 0.002,
        max_stop_loss: float = 0.10,
        min_take_profit: float = 0.002,
        max_take_profit: float = 0.20,
        min_duration: int = 1,
        max_duration: int = 100,
        include_history_position_features: bool = True,
        close_position_on_opposite_signal: bool = True,
        terminate_on_bankrupt: bool = True,
        bankrupt_threshold: float = 0.10,
        eps: float = 1e-8,
    ):
        super().__init__()

        assert len(data) == len(prices), "data y prices deben tener la misma longitud"
        assert len(prices) > 2, "Se necesitan al menos 3 timesteps"
        assert initial_budget > 0
        assert 0 <= trade_fee < 1

        self.data = np.asarray(data, dtype=np.float32)
        self.prices = np.asarray(prices, dtype=np.float32)
        self.n_steps = len(prices)
        self.n_features = self.data.shape[1] if self.data.ndim > 1 else 1

        if self.data.ndim == 1:
            self.data = self.data.reshape(-1, 1)

        self.initial_budget = float(initial_budget)
        self.trade_fee = float(trade_fee)
        self.allow_short = allow_short
        self.reward_scaling = reward_scaling
        self.drawdown_penalty = drawdown_penalty
        self.unrealized_pnl_weight = unrealized_pnl_weight
        self.min_stop_loss = min_stop_loss
        self.max_stop_loss = max_stop_loss
        self.min_take_profit = min_take_profit
        self.max_take_profit = max_take_profit
        self.min_duration = min_duration
        self.max_duration = max_duration
        self.include_history_position_features = include_history_position_features
        self.close_position_on_opposite_signal = close_position_on_opposite_signal
        self.terminate_on_bankrupt = terminate_on_bankrupt
        self.bankrupt_threshold = bankrupt_threshold
        self.eps = eps

        self.max_episode_steps = max_episode_steps or (self.n_steps - 1)

        # Acción:
        # [action_type_continuous, size, stop_loss, take_profit, duration_norm]
        #
        # action_type_continuous:
        #   [0.0, 1/3)   -> HOLD
        #   [1/3, 2/3)   -> BUY
        #   [2/3, 1.0]   -> SELL
        #
        # size: [0,1]
        # stop_loss: [0,1] -> escalado a [min_stop_loss, max_stop_loss]
        # take_profit: [0,1] -> escalado a [min_take_profit, max_take_profit]
        # duration_norm: [0,1] -> escalado a [min_duration, max_duration]
        self.action_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

        # Observación:
        # [features..., current_price,
        #  budget, quantity, side, entry_price,
        #  stop_loss_price, take_profit_price,
        #  max_duration, age,
        #  unrealized_pnl, realized_pnl,
        #  equity, peak_equity, drawdown,
        #  free_cash_ratio, position_value_ratio]
        obs_dim = self.n_features + 16
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32,
        )

        self._reset_internal_state()

    def _reset_internal_state(self):
        self.current_step = 0
        self.steps_in_episode = 0

        self.cash = self.initial_budget
        self.realized_pnl = 0.0
        self.position = Position()
        self.last_equity = self.initial_budget
        self.equity = self.initial_budget
        self.peak_equity = self.initial_budget

        self.total_fees_paid = 0.0
        self.total_trades = 0
        self.closed_trades = 0
        self.winning_trades = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_internal_state()
        obs = self._get_observation()
        info = self._get_info()
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)
        assert action.shape == (5,)

        prev_equity = self._mark_to_market_equity(self._current_price())

        action_type = self._decode_action_type(action[0])
        size = float(np.clip(action[1], 0.0, 1.0))
        stop_loss_pct = self._scale(action[2], self.min_stop_loss, self.max_stop_loss)
        take_profit_pct = self._scale(action[3], self.min_take_profit, self.max_take_profit)
        duration = int(round(self._scale(action[4], self.min_duration, self.max_duration)))

        # 1) Revisar si la posición abierta debe cerrarse por SL/TP/duración
        auto_closed = self._check_exit_conditions()

        # 2) Aplicar acción del agente
        if action_type == self.HOLD:
            pass
        elif action_type == self.BUY:
            self._execute_buy(size, stop_loss_pct, take_profit_pct, duration)
        elif action_type == self.SELL:
            self._execute_sell(size, stop_loss_pct, take_profit_pct, duration)

        # 3) Envejecer posición abierta
        if self.position.side != 0:
            self.position.age += 1

        # 4) Avanzar tiempo
        self.current_step += 1
        self.steps_in_episode += 1

        terminated = self.current_step >= (self.n_steps - 1)
        truncated = self.steps_in_episode >= self.max_episode_steps

        current_equity = self._mark_to_market_equity(self._current_price())
        self.equity = current_equity
        self.peak_equity = max(self.peak_equity, self.equity)

        drawdown = self._compute_drawdown()

        if self.terminate_on_bankrupt:
            if self.equity <= self.initial_budget * self.bankrupt_threshold:
                terminated = True

        reward = self._compute_reward(prev_equity, current_equity, drawdown)

        obs = self._get_observation()
        info = self._get_info()
        info["auto_closed"] = auto_closed

        self.last_equity = current_equity
        return obs, reward, terminated, truncated, info

    def render(self):
        print(
            f"step={self.current_step} "
            f"price={self._current_price():.4f} "
            f"cash={self.cash:.2f} "
            f"qty={self.position.quantity:.6f} "
            f"side={self.position.side} "
            f"entry={self.position.entry_price:.4f} "
            f"equity={self.equity:.2f}"
        )

    # =========================
    # Core helpers
    # =========================

    def _current_price(self) -> float:
        return float(self.prices[self.current_step])

    def _decode_action_type(self, x: float) -> int:
        if x < 1.0 / 3.0:
            return self.HOLD
        elif x < 2.0 / 3.0:
            return self.BUY
        return self.SELL

    def _scale(self, x: float, low: float, high: float) -> float:
        x = float(np.clip(x, 0.0, 1.0))
        return low + x * (high - low)

    def _position_market_value(self, price: float) -> float:
        if self.position.side == 0 or self.position.quantity <= 0:
            return 0.0
        return self.position.quantity * price

    def _unrealized_pnl(self, price: float) -> float:
        if self.position.side == 0 or self.position.quantity <= 0:
            return 0.0

        if self.position.side == 1:
            return (price - self.position.entry_price) * self.position.quantity
        else:
            return (self.position.entry_price - price) * self.position.quantity

    def _mark_to_market_equity(self, price: float) -> float:
        # Para long:
        #   equity = cash + market_value
        # Para short:
        #   usamos cash + unrealized_pnl como aproximación operativa.
        if self.position.side == 1:
            return self.cash + self._position_market_value(price)
        elif self.position.side == -1:
            return self.cash + self._unrealized_pnl(price)
        return self.cash

    def _compute_drawdown(self) -> float:
        if self.peak_equity <= self.eps:
            return 0.0
        return max(0.0, (self.peak_equity - self.equity) / self.peak_equity)

    def _compute_reward(self, prev_equity: float, current_equity: float, drawdown: float) -> float:
        equity_delta = current_equity - prev_equity
        reward = equity_delta * self.reward_scaling

        if self.unrealized_pnl_weight != 0.0:
            reward += self.unrealized_pnl_weight * self._unrealized_pnl(self._current_price())

        if self.drawdown_penalty != 0.0:
            reward -= self.drawdown_penalty * drawdown

        return float(reward)

    # =========================
    # Trading logic
    # =========================

    def _check_exit_conditions(self) -> bool:
        if self.position.side == 0:
            return False

        price = self._current_price()
        should_close = False

        if self.position.side == 1:
            if price <= self.position.stop_loss_price:
                should_close = True
            elif price >= self.position.take_profit_price:
                should_close = True
        elif self.position.side == -1:
            if price >= self.position.stop_loss_price:
                should_close = True
            elif price <= self.position.take_profit_price:
                should_close = True

        if self.position.age >= self.position.max_duration:
            should_close = True

        if should_close:
            self._close_position(price)
            return True

        return False

    def _execute_buy(self, size: float, stop_loss_pct: float, take_profit_pct: float, duration: int):
        price = self._current_price()

        # Si hay short abierto, opcionalmente cerrarlo antes
        if self.position.side == -1 and self.close_position_on_opposite_signal:
            self._close_position(price)

        # Si ya hay long, actualizamos/expandimos
        if self.position.side in (0, 1):
            notional = self.cash * size
            if notional <= self.eps:
                return

            fee = notional * self.trade_fee
            effective_notional = max(0.0, notional - fee)
            qty = effective_notional / price

            if qty <= self.eps:
                return

            if self.position.side == 0:
                self.cash -= notional
                self.total_fees_paid += fee
                self.total_trades += 1

                self.position = Position(
                    side=1,
                    quantity=qty,
                    entry_price=price,
                    stop_loss_price=price * (1.0 - stop_loss_pct),
                    take_profit_price=price * (1.0 + take_profit_pct),
                    max_duration=duration,
                    age=0,
                )
            else:
                total_cost_basis = self.position.entry_price * self.position.quantity + price * qty
                new_qty = self.position.quantity + qty
                avg_entry = total_cost_basis / max(new_qty, self.eps)

                self.cash -= notional
                self.total_fees_paid += fee
                self.total_trades += 1

                self.position.quantity = new_qty
                self.position.entry_price = avg_entry
                self.position.stop_loss_price = avg_entry * (1.0 - stop_loss_pct)
                self.position.take_profit_price = avg_entry * (1.0 + take_profit_pct)
                self.position.max_duration = duration
                self.position.age = 0

    def _execute_sell(self, size: float, stop_loss_pct: float, take_profit_pct: float, duration: int):
        price = self._current_price()

        # Si hay long abierto, opcionalmente cerrarlo antes
        if self.position.side == 1 and self.close_position_on_opposite_signal:
            self._close_position(price)

        # Abrir o ampliar short
        if not self.allow_short:
            return

        if self.position.side in (0, -1):
            notional = self.cash * size
            if notional <= self.eps:
                return

            fee = notional * self.trade_fee
            qty = notional / price

            if qty <= self.eps:
                return

            # Aproximación práctica para short:
            # reservamos fee, pero no inmovilizamos todo el notional como margin exacto.
            # Para más realismo se puede agregar initial_margin_ratio.
            self.cash -= fee
            self.total_fees_paid += fee
            self.total_trades += 1

            if self.position.side == 0:
                self.position = Position(
                    side=-1,
                    quantity=qty,
                    entry_price=price,
                    stop_loss_price=price * (1.0 + stop_loss_pct),
                    take_profit_price=price * (1.0 - take_profit_pct),
                    max_duration=duration,
                    age=0,
                )
            else:
                total_cost_basis = self.position.entry_price * self.position.quantity + price * qty
                new_qty = self.position.quantity + qty
                avg_entry = total_cost_basis / max(new_qty, self.eps)

                self.position.quantity = new_qty
                self.position.entry_price = avg_entry
                self.position.stop_loss_price = avg_entry * (1.0 + stop_loss_pct)
                self.position.take_profit_price = avg_entry * (1.0 - take_profit_pct)
                self.position.max_duration = duration
                self.position.age = 0

    def _close_position(self, price: float):
        if self.position.side == 0 or self.position.quantity <= self.eps:
            return

        qty = self.position.quantity

        if self.position.side == 1:
            notional = qty * price
            fee = notional * self.trade_fee
            pnl = (price - self.position.entry_price) * qty - fee

            self.cash += notional - fee
            self.realized_pnl += pnl
            self.total_fees_paid += fee

        elif self.position.side == -1:
            notional = qty * price
            fee = notional * self.trade_fee
            pnl = (self.position.entry_price - price) * qty - fee

            self.cash += pnl
            self.realized_pnl += pnl
            self.total_fees_paid += fee

        self.closed_trades += 1
        if pnl > 0:
            self.winning_trades += 1

        self.position = Position()

    # =========================
    # Observation / info
    # =========================

    def _get_observation(self) -> np.ndarray:
        price = self._current_price()
        unrealized = self._unrealized_pnl(price)
        equity = self._mark_to_market_equity(price)
        peak_equity = max(self.peak_equity, equity)
        drawdown = 0.0 if peak_equity <= self.eps else max(0.0, (peak_equity - equity) / peak_equity)

        budget = self.cash
        quantity = self.position.quantity
        side = float(self.position.side)
        entry_price = self.position.entry_price
        sl_price = self.position.stop_loss_price
        tp_price = self.position.take_profit_price
        max_dur = float(self.position.max_duration)
        age = float(self.position.age)

        free_cash_ratio = budget / max(self.initial_budget, self.eps)
        position_value_ratio = self._position_market_value(price) / max(self.initial_budget, self.eps)

        obs = np.concatenate([
            self.data[self.current_step].astype(np.float32),
            np.array([
                price,
                budget,
                quantity,
                side,
                entry_price,
                sl_price,
                tp_price,
                max_dur,
                age,
                unrealized,
                self.realized_pnl,
                equity,
                peak_equity,
                drawdown,
                free_cash_ratio,
                position_value_ratio,
            ], dtype=np.float32)
        ])

        return obs.astype(np.float32)

    def _get_info(self) -> Dict[str, Any]:
        price = self._current_price()
        unrealized = self._unrealized_pnl(price)
        equity = self._mark_to_market_equity(price)
        drawdown = self._compute_drawdown()
        win_rate = self.winning_trades / self.closed_trades if self.closed_trades > 0 else 0.0

        return {
            "step": self.current_step,
            "price": price,
            "cash": self.cash,
            "equity": equity,
            "realized_pnl": self.realized_pnl,
            "unrealized_pnl": unrealized,
            "drawdown": drawdown,
            "position_side": self.position.side,
            "position_quantity": self.position.quantity,
            "position_entry_price": self.position.entry_price,
            "position_stop_loss": self.position.stop_loss_price,
            "position_take_profit": self.position.take_profit_price,
            "position_age": self.position.age,
            "position_max_duration": self.position.max_duration,
            "total_fees_paid": self.total_fees_paid,
            "total_trades": self.total_trades,
            "closed_trades": self.closed_trades,
            "winning_trades": self.winning_trades,
            "win_rate": win_rate,
        }

In [4]:
n = 1000
n_features = 8

data = np.random.randn(n, n_features).astype(np.float32)
prices = (100 + np.cumsum(np.random.randn(n))).astype(np.float32)

In [5]:
env = TradingRLEnv(
    data=data,
    prices=prices,
    initial_budget=10000,
    trade_fee=0.001,
    allow_short=True,
    reward_scaling=0.01,
    drawdown_penalty=0.1,
    min_stop_loss=0.003,
    max_stop_loss=0.05,
    min_take_profit=0.005,
    max_take_profit=0.10,
    min_duration=1,
    max_duration=50,
)



In [6]:
obs, info = env.reset()

for _ in range(20):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()
    if terminated or truncated:
        break

step=1 price=102.7729 cash=10000.00 qty=0.000000 side=0 entry=0.0000 equity=10000.00
step=2 price=100.2469 cash=3088.63 qty=67.181669 side=1 entry=102.7729 equity=9823.39
step=3 price=99.3351 cash=3088.63 qty=67.181669 side=1 entry=102.7729 equity=9762.13
step=4 price=99.9081 cash=7056.52 qty=27.142858 side=1 entry=99.3351 equity=9768.31
step=5 price=98.5680 cash=9759.90 qty=57.000493 side=-1 entry=99.9081 equity=9836.28
step=6 price=98.1839 cash=9830.67 qty=0.000000 side=0 entry=0.0000 equity=9830.67
step=7 price=96.4017 cash=4589.07 qty=53.332153 side=1 entry=98.1839 equity=9730.38
step=8 price=95.7247 cash=1336.98 qty=87.033181 side=1 entry=97.4938 equity=9668.20
step=9 price=94.7714 cash=5181.65 qty=46.735515 side=1 entry=95.7247 equity=9610.84
step=10 price=93.1221 cash=9601.41 qty=52.763371 side=-1 entry=94.7714 equity=9688.44
step=11 price=94.8904 cash=9601.41 qty=52.763371 side=-1 entry=94.7714 equity=9595.14
step=12 price=96.2270 cash=9601.41 qty=52.763371 side=-1 entry=94.771

In [7]:
action

array([0.5549882 , 0.4563091 , 0.71705425, 0.99869615, 0.87480986],
      dtype=float32)

In [8]:
obs

array([ 1.5297210e+00,  4.3716806e-01,  4.5853975e-01,  2.8621923e-02,
        4.4014998e-02,  1.6354746e-01, -2.4949215e-01, -2.8130934e-01,
        9.5035965e+01,  5.2042344e+03,  4.5579819e+01,  1.0000000e+00,
        9.5731934e+01,  9.2218422e+01,  1.0529327e+02,  4.4000000e+01,
        1.0000000e+00, -3.1722124e+01, -3.7592807e+02,  9.5359561e+03,
        1.0000000e+04,  4.6404373e-02,  5.2042341e-01,  4.3317220e-01],
      dtype=float32)